## install & load required packages

In [ ]:
required_pkgs <- c("readr", "jsonlite", "readxl", "writexl",
                    "dplyr", "tidyr", "DBI", "RSQLite", "lubridate")

new_pkgs <- required_pkgs[!(required_pkgs %in% installed.packages()[, "Package"])]
if (length(new_pkgs)) install.packages(new_pkgs)

lapply(required_pkgs, library, character.only = TRUE)

set.seed(42)

Installing packages into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)


Attaching package: ‘dplyr’


The following objects are masked from ‘package:stats’:

    filter, lag


The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union



Attaching package: ‘lubridate’


The following objects are masked from ‘package:base’:

    date, intersect, setdiff, union




[[1]]
[1] "readr"     "stats"     "graphics"  "grDevices" "utils"     "datasets" 
[7] "methods"   "base"     

[[2]]
[1] "jsonlite"  "readr"     "stats"     "graphics"  "grDevices" "utils"    
[7] "datasets"  "methods"   "base"     

[[3]]
 [1] "readxl"    "jsonlite"  "readr"     "stats"     "graphics"  "grDevices"
 [7] "utils"     "datasets"  "methods"   "base"     

[[4]]
 [1] "writexl"   "readxl"    "jsonlite"  "readr"     "stats"     "graphics" 
 [7] "grDevices" "utils"     "datasets"  "methods"   "base"     

[[5]]
 [1] "dplyr"     "writexl"   "readxl"    "jsonlite"  "readr"     "stats"    
 [7] "graphics"  "grDevices" "utils"     "datasets"  "methods"   "base"     

[[6]]
 [1] "tidyr"     "dplyr"     "writexl"   "readxl"    "jsonlite"  "readr"    
 [7] "stats"     "graphics"  "grDevices" "utils"     "datasets"  "methods"  
[13] "base"     

[[7]]
 [1] "DBI"       "tidyr"     "dplyr"     "writexl"   "readxl"    "jsonlite" 
 [7] "readr"     "stats"     "graphics"  "grDevices" "utils"     "datasets" 
[13] "methods"   "base"     

[[8]]
 [1] "RSQLite"   "DBI"       "tidyr"     "dplyr"     "writexl"   "readxl"   
 [7] "jsonlite"  "readr"     "stats"     "graphics"  "grDevices" "utils"    
[13] "datasets"  "methods"   "base"     

[[9]]
 [1] "lubridate" "RSQLite"   "DBI"       "tidyr"     "dplyr"     "writexl"  
 [7] "readxl"    "jsonlite"  "readr"     "stats"     "graphics"  "grDevices"
[13] "utils"     "datasets"  "methods"   "base"

## 1. GET THE DATA

In [ ]:
uci_url <- "https://archive.ics.uci.edu/ml/machine-learning-databases/00352/Online%20Retail.xlsx"
raw_path <- "OnlineRetail_raw.xlsx"

download_ok <- tryCatch({
  download.file(uci_url, destfile = raw_path, mode = "wb", quiet = TRUE)
  TRUE
}, error = function(e) FALSE)

if (download_ok && file.exists(raw_path)) {

  message("UCI dataset downloaded successfully. Building source files from it...")
  raw <- read_excel(raw_path)

  # Standardise column names to match the problem statement
  raw <- raw %>%
    rename(
      InvoiceNo   = InvoiceNo,
      StockCode   = StockCode,
      Description = Description,
      Quantity    = Quantity,
      InvoiceDate = InvoiceDate,
      UnitPrice   = UnitPrice,
      CustomerID  = CustomerID,
      Country     = Country
    )

  # transactions.csv
  transactions_src <- raw %>%
    select(InvoiceNo, StockCode, CustomerID, Quantity, InvoiceDate)
  write_csv(transactions_src, "transactions.csv")

  # products.json  (one row per unique StockCode)
  products_src <- raw %>%
    group_by(StockCode) %>%
    summarise(Description = first(na.omit(Description)),
              UnitPrice   = first(na.omit(UnitPrice)), .groups = "drop")
  write_json(products_src, "products.json", auto_unbox = TRUE, pretty = TRUE)

  # customers.xlsx (one row per unique CustomerID)
  customers_src <- raw %>%
    filter(!is.na(CustomerID)) %>%
    distinct(CustomerID, Country)
  write_xlsx(customers_src, "customers.xlsx")

} else {

  message("Could not reach the UCI repository — generating a synthetic ",
          "sample dataset with the same schema so the pipeline still runs.")

  n_cust  <- 200
  n_prod  <- 60
  n_trans <- 5000

  countries <- c("United Kingdom", "Germany", "France", "EIRE", "Spain",
                 "Netherlands", "Belgium", "Portugal", "Australia", "Italy")

  customers_src <- data.frame(
    CustomerID = 10000:(10000 + n_cust - 1),
    Country    = sample(countries, n_cust, replace = TRUE,
                         prob = c(0.45,0.08,0.07,0.06,0.06,0.06,0.05,0.05,0.06,0.06))
  )
  write_xlsx(customers_src, "customers.xlsx")

  products_src <- data.frame(
    StockCode   = sprintf("P%04d", 1:n_prod),
    Description = paste("Product", 1:n_prod),
    UnitPrice   = round(runif(n_prod, 0.5, 50), 2)
  )
  # inject a few bad prices to exercise the cleaning step
  products_src$UnitPrice[sample(1:n_prod, 3)] <- 0
  write_json(products_src, "products.json", auto_unbox = TRUE, pretty = TRUE)

  transactions_src <- data.frame(
    InvoiceNo   = sprintf("INV%06d", sample(1:2000, n_trans, replace = TRUE)),
    StockCode   = sample(products_src$StockCode, n_trans, replace = TRUE),
    CustomerID  = sample(c(customers_src$CustomerID, NA), n_trans, replace = TRUE,
                          prob = c(rep(0.95/n_cust, n_cust), 0.05)),
    Quantity    = sample(c(-5:-1, 1:20), n_trans, replace = TRUE),  # includes invalid negatives
    InvoiceDate = format(Sys.Date() - sample(0:365, n_trans, replace = TRUE), "%Y-%m-%d")
  )
  # inject some duplicates and zero quantities
  transactions_src <- bind_rows(transactions_src, transactions_src[sample(1:n_trans, 50), ])
  transactions_src$Quantity[sample(1:nrow(transactions_src), 20)] <- 0
  write_csv(transactions_src, "transactions.csv")
}

UCI dataset downloaded successfully. Building source files from it...



## TASK 1: IMPORT AND CLEAN THE DATA

In [ ]:
# ---- Import ----------------------------------------------------------
transactions <- read_csv("transactions.csv", show_col_types = FALSE)
products     <- fromJSON("products.json") %>% as_tibble()
customers    <- read_excel("customers.xlsx")

cat("Raw row counts -> transactions:", nrow(transactions),
    "| products:", nrow(products),
    "| customers:", nrow(customers), "\n")

# ---- Inspect -----------------------------------------------------------
glimpse(transactions)
glimpse(products)
glimpse(customers)

sapply(transactions, function(x) sum(is.na(x)))
sapply(products, function(x) sum(is.na(x)))
sapply(customers, function(x) sum(is.na(x)))

# ---- Clean: transactions -----------------------------------------------
# Cleaning decisions:
# 1. Drop rows with missing CustomerID or StockCode (can't attribute revenue).
# 2. Remove exact duplicate rows.
# 3. Remove rows with zero or negative Quantity (returns/cancellations are
#    out of scope for this "sales performance" analysis).
# 4. Parse InvoiceDate to a proper Date type.

transactions_clean <- transactions %>%
  distinct() %>%
  filter(!is.na(CustomerID), !is.na(StockCode)) %>%
  filter(Quantity > 0) %>%
  mutate(InvoiceDate = as.Date(InvoiceDate))

cat("Transactions after cleaning:", nrow(transactions_clean),
    "(removed", nrow(transactions) - nrow(transactions_clean), "rows)\n")

# ---- Clean: products -----------------------------------------------------
# Cleaning decisions:
# 1. Drop duplicate StockCodes (keep first).
# 2. Remove products with missing or zero/negative UnitPrice — price of 0
#    is treated as a data-entry error, not a genuine free item.

products_clean <- products %>%
  distinct(StockCode, .keep_all = TRUE) %>%
  filter(!is.na(UnitPrice), UnitPrice > 0)

cat("Products after cleaning:", nrow(products_clean),
    "(removed", nrow(products) - nrow(products_clean), "rows)\n")

# ---- Clean: customers -----------------------------------------------------
# Cleaning decisions:
# 1. Drop duplicate CustomerIDs.
# 2. Drop rows with missing Country.

customers_clean <- customers %>%
  distinct(CustomerID, .keep_all = TRUE) %>%
  filter(!is.na(Country))

cat("Customers after cleaning:", nrow(customers_clean),
    "(removed", nrow(customers) - nrow(customers_clean), "rows)\n")

Raw row counts -> transactions: 541909 | products: 4070 | customers: 4380 
Rows: 541,909
Columns: 5
$ InvoiceNo   <chr> "536365", "536365", "536365", "536365", "536365", "536365"…
$ StockCode   <chr> "85123A", "71053", "84406B", "84029G", "84029E", "22752", …
$ CustomerID  <dbl> 17850, 17850, 17850, 17850, 17850, 17850, 17850, 17850, 17…
$ Quantity    <dbl> 6, 6, 8, 6, 6, 2, 6, 6, 6, 32, 6, 6, 8, 6, 6, 3, 2, 3, 3, …
$ InvoiceDate <dttm> 2010-12-01 08:26:00, 2010-12-01 08:26:00, 2010-12-01 08:2…
Rows: 4,070
Columns: 3
$ StockCode   <chr> "10002", "10080", "10120", "10123C", "10123G", "10124A", "…
$ Description <chr> "INFLATABLE POLITICAL GLOBE", "GROOVY CACTUS INFLATABLE", …
$ UnitPrice   <dbl> 0.85, 0.85, 0.21, 0.65, 0.00, 0.42, 0.42, 0.85, 0.85, 0.00…
Rows: 4,380
Columns: 2
$ CustomerID <dbl> 17850, 13047, 12583, 13748, 15100, 15291, 14688, 17809, 153…
$ Country    <chr> "United Kingdom", "United Kingdom", "France", "United Kingd…


InvoiceNo   StockCode  CustomerID    Quantity InvoiceDate 
          0           0      135080           0           0

StockCode Description   UnitPrice 
          0         112           0

CustomerID    Country 
         0          0

Transactions after cleaning: 392708 (removed 149201 rows)
Products after cleaning: 3855 (removed 215 rows)
Customers after cleaning: 4372 (removed 8 rows)


## TASK 2: INTEGRATE THE MULTIPLE DATA SOURCES

In [ ]:
# We use inner_join() for products because a transaction line with no
# matching (clean) product has no reliable UnitPrice and therefore no
# computable Revenue — keeping it would force an NA into every downstream
# aggregation. We use left_join() for customers because we still want to
# retain sales rows even if a customer's demographic record is incomplete;
# we'd rather see "unmatched" customer info than silently drop revenue.

sales_data <- transactions_clean %>%
  inner_join(products_clean, by = "StockCode") %>%
  left_join(customers_clean, by = "CustomerID") %>%
  mutate(Revenue = Quantity * UnitPrice)

cat("Final integrated dataset dimensions:", dim(sales_data)[1], "rows x",
    dim(sales_data)[2], "cols\n")

# Unmatched records check
unmatched_products <- anti_join(transactions_clean, products_clean, by = "StockCode")
unmatched_customers <- anti_join(transactions_clean, customers_clean, by = "CustomerID")

cat("Transaction rows with no matching product (excluded by inner_join):",
    nrow(unmatched_products), "\n")
cat("Transaction rows with no matching customer record (Country will be NA):",
    nrow(unmatched_customers), "\n")

# Drop any residual rows without a Country, since country-level analysis
# (Task 3) needs it
sales_data <- sales_data %>% filter(!is.na(Country))

Final integrated dataset dimensions: 387877 rows x 9 cols
Transaction rows with no matching product (excluded by inner_join): 4831 
Transaction rows with no matching customer record (Country will be NA): 0 


## TASK 3: SALES AND CUSTOMER ANALYSIS

In [ ]:
# ---- 1. Total sales revenue --------------------------------------------
total_revenue <- sum(sales_data$Revenue)
cat("Total Sales Revenue:", round(total_revenue, 2), "\n")

# ---- 2. Top 5 products by revenue --------------------------------------
top5_products <- sales_data %>%
  group_by(StockCode, Description) %>%
  summarise(TotalRevenue = sum(Revenue), .groups = "drop") %>%
  arrange(desc(TotalRevenue)) %>%
  slice_head(n = 5)
print(top5_products)

# ---- 3. Top 5 countries by revenue --------------------------------------
top5_countries <- sales_data %>%
  group_by(Country) %>%
  summarise(TotalRevenue = sum(Revenue), .groups = "drop") %>%
  arrange(desc(TotalRevenue)) %>%
  slice_head(n = 5)
print(top5_countries)

# ---- 4. Top 5 customers by purchase value --------------------------------
top5_customers <- sales_data %>%
  group_by(CustomerID) %>%
  summarise(TotalRevenue = sum(Revenue), .groups = "drop") %>%
  arrange(desc(TotalRevenue)) %>%
  slice_head(n = 5)
print(top5_customers)

# ---- Customer value classification (case_when) ---------------------------
customer_value <- sales_data %>%
  group_by(CustomerID) %>%
  summarise(TotalRevenue = sum(Revenue), .groups = "drop") %>%
  mutate(
    ValueSegment = case_when(
      TotalRevenue < 100                          ~ "Low Value",
      TotalRevenue >= 100  & TotalRevenue < 500    ~ "Medium Value",
      TotalRevenue >= 500  & TotalRevenue < 2000   ~ "High Value",
      TotalRevenue >= 2000                         ~ "Premium",
      TRUE                                         ~ "Unclassified"
    )
  )

print(table(customer_value$ValueSegment))

# Merge segment back into the main dataset (handy for the SQLite export)
sales_data <- sales_data %>%
  left_join(customer_value %>% select(CustomerID, ValueSegment), by = "CustomerID")

# ---- High-performing vs underperforming market ---------------------------
market_perf <- sales_data %>%
  group_by(Country) %>%
  summarise(TotalRevenue = sum(Revenue), Orders = n_distinct(InvoiceNo),
            .groups = "drop") %>%
  arrange(desc(TotalRevenue))

best_market  <- market_perf %>% slice_head(n = 1)
worst_market <- market_perf %>% slice_tail(n = 1)

cat("\nHigh-performing market:", best_market$Country,
    "-> Revenue:", round(best_market$TotalRevenue, 2),
    "| Orders:", best_market$Orders, "\n")
cat("Underperforming market:", worst_market$Country,
    "-> Revenue:", round(worst_market$TotalRevenue, 2),
    "| Orders:", worst_market$Orders, "\n")

Total Sales Revenue: 10752840 
# A tibble: 5 × 3
  StockCode Description                        TotalRevenue
  <chr>     <chr>                                     <dbl>
1 23843     PAPER CRAFT , LITTLE BIRDIE             168470.
2 47566     PARTY BUNTING                           142438.
3 22423     REGENCY CAKESTAND 3 TIER                135605.
4 85123A    WHITE HANGING HEART T-LIGHT HOLDER       93746.
5 23166     MEDIUM CERAMIC TOP STORAGE JAR           81033.
# A tibble: 5 × 2
  Country        TotalRevenue
  <chr>                 <dbl>
1 United Kingdom     8861857.
2 Netherlands         363884.
3 EIRE                331660.
4 Germany             263819.
5 France              226976.
# A tibble: 5 × 2
  CustomerID TotalRevenue
       <dbl>        <dbl>
1      18102      408760.
2      14646      357531.
3      17450      186038.
4      14911      182691.
5      16446      168472.

  High Value    Low Value Medium Value      Premium 
        1738          117         1399         10

## TASK 4: STORE AND RETRIEVE DATA USING SQL

In [ ]:
con <- dbConnect(RSQLite::SQLite(), "retail_analysis.sqlite")

dbWriteTable(con, "retail_sales", sales_data, overwrite = TRUE)

# Query 1: Top 5 customers by revenue
q1 <- dbGetQuery(con, "
  SELECT CustomerID, SUM(Revenue) AS TotalRevenue
  FROM retail_sales
  GROUP BY CustomerID
  ORDER BY TotalRevenue DESC
  LIMIT 5;
")
cat("\nSQL Query 1 - Top 5 customers by revenue:\n")
print(q1)

# Query 2: Total revenue by country
q2 <- dbGetQuery(con, "
  SELECT Country, SUM(Revenue) AS TotalRevenue
  FROM retail_sales
  GROUP BY Country
  ORDER BY TotalRevenue DESC;
")
cat("\nSQL Query 2 - Total revenue by country:\n")
print(q2)

dbDisconnect(con)


SQL Query 1 - Top 5 customers by revenue:
  CustomerID TotalRevenue
1      18102     408760.0
2      14646     357531.1
3      17450     186038.0
4      14911     182690.5
5      16446     168472.5

SQL Query 2 - Total revenue by country:
                Country TotalRevenue
1        United Kingdom   8861857.13
2           Netherlands    363884.48
3                  EIRE    331660.17
4               Germany    263818.97
5                France    226975.60
6             Australia    173918.61
7                 Spain     67426.09
8           Switzerland     66619.97
9                 Japan     48600.22
10              Belgium     47858.02
11               Sweden     43652.09
12               Norway     40283.20
13             Portugal     32679.45
14              Finland     23306.79
15      Channel Islands     22059.72
16              Denmark     21291.79
17                Italy     21065.45
18               Cyprus     16829.62
19            Singapore     11497.98
20               Pol

## Conclusion

1. Revenue Concentration in Key Markets: Revenue is heavily concentrated in a small number of top markets (see top5_countries). Expansion or marketing budgets should weigh toward these markets while testing targeted campaigns in weaker ones.

2. 80/20 Rule in Revenue Distribution: The top 5 products and customers contribute a disproportionate share of total revenue. Retention programs specifically designed for "Premium" and "High Value" segments are likely to yield an outsized ROI.

3. Data Quality & Pipeline Integrity: A meaningful share of transaction rows had to be excluded during data cleaning/integration due to missing IDs, zero quantities, or unmatched products. Tightening data capture rules at the point of sale will significantly improve the reliability of future analyses.